In [1]:
"""
Shape Recognizer - draw a shape on a coordinate grid, get its equation back.

Draw with the left mouse button. When you release, the app fits the
stroke to a line / circle / ellipse / parabola and shows the best match
with its equation, overlaying the fitted curve on your sketch.
"""

import tkinter as tk
from tkinter import font as tkfont
import numpy as np

# Removed: from fitting import recognize_shape
# The function 'recognize_shape' must be defined in the notebook (Cell DM605bMGoaLx)

class ShapeRecognizerApp:
    def __init__(self, root, width=900, height=650, scale=25):
        self.root = root
        self.root.title("Shape Recognizer")

        self.width, self.height, self.scale = width, height, scale
        self.origin = (width // 2, height // 2)

        # ---------- layout ----------
        top = tk.Frame(root)
        top.pack(side=tk.TOP, fill=tk.X, padx=8, pady=6)

        self.result_font = tkfont.Font(family="Menlo", size=13, weight="bold")
        self.result_var = tk.StringVar(value="Draw a shape on the grid \u2192")
        tk.Label(top, textvariable=self.result_var, font=self.result_font,
                 anchor="w", fg="#1a5276").pack(side=tk.LEFT, fill=tk.X, expand=True)

        tk.Button(top, text="Clear", command=self.clear).pack(side=tk.RIGHT, padx=4)
        tk.Button(top, text="Zoom +", command=lambda: self.rescale(1.25)).pack(side=tk.RIGHT, padx=4)
        tk.Button(top, text="Zoom -", command=lambda: self.rescale(0.8)).pack(side=tk.RIGHT, padx=4)

        self.canvas = tk.Canvas(root, width=width, height=height, bg="white",
                                 highlightthickness=0)
        self.canvas.pack(side=tk.TOP, fill=tk.BOTH, expand=True)

        self.canvas.bind("<ButtonPress-1>", self.on_press)
        self.canvas.bind("<B1-Motion>", self.on_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_release)
        self.root.bind("<Configure>", self._on_resize)

        self.points = []       # current stroke, math-space coords
        self.last_pixel = None

        self.draw_axes()

    # ---------------- coordinate system ----------------

    def draw_axes(self):
        self.canvas.delete("axes")
        w, h = self.canvas.winfo_width() or self.width, self.canvas.winfo_height() or self.height
        ox, oy = self.origin

        # grid lines every unit
        for gx in range(ox % self.scale, w, self.scale):
            self.canvas.create_line(gx, 0, gx, h, fill="#eeeeee", tags="axes")
        for gy in range(oy % self.scale, h, self.scale):
            self.canvas.create_line(0, gy, w, gy, fill="#eeeeee", tags="axes")

        # axes
        self.canvas.create_line(0, oy, w, oy, fill="#888888", width=2, tags="axes")
        self.canvas.create_line(ox, 0, ox, h, fill="#888888", width=2, tags="axes")

        # tick labels every 2 units
        step = 2
        for i in range(-ox // self.scale, (w - ox) // self.scale + 1, step):
            if i == 0:
                continue
            x = ox + i * self.scale
            self.canvas.create_line(x, oy - 4, x, oy + 4, fill="#888888", tags="axes")
            self.canvas.create_text(x, oy + 12, text=str(i), fill="#888888",
                                     font=("Helvetica", 8), tags="axes")
        for i in range(-oy // self.scale, (h - oy) // self.scale + 1, step):
            if i == 0:
                continue
            y = oy - i * self.scale
            self.canvas.create_line(ox - 4, y, ox + 4, y, fill="#888888", tags="axes")
            self.canvas.create_text(ox - 14, y, text=str(i), fill="#888888",
                                     font=("Helvetica", 8), tags="axes")

        self.canvas.tag_lower("axes")

    def _on_resize(self, event):
        # keep origin centered when the window is resized
        if event.widget == self.root:
            w, h = self.canvas.winfo_width(), self.canvas.winfo_height()
            if w > 10 and h > 10:
                self.origin = (w // 2, h // 2)
                self.draw_axes()

    def rescale(self, factor):
        self.scale = max(5, min(100, int(self.scale * factor)))
        self.draw_axes()

    def to_math(self, px, py):
        ox, oy = self.origin
        return (px - ox) / self.scale, (oy - py) / self.scale

    def to_pixel(self, x, y):
        ox, oy = self.origin
        return ox + x * self.scale, oy - y * self.scale

    # ---------------- drawing interaction ----------------

    def on_press(self, event):
        self.canvas.delete("stroke")
        self.canvas.delete("fit")
        self.points = []
        self.last_pixel = (event.x, event.y)
        self._add_point(event.x, event.y)
        self.result_var.set("Drawing...")

    def on_drag(self, event):
        lx, ly = self.last_pixel
        self.canvas.create_line(lx, ly, event.x, event.y, fill="#2e86c1",
                                 width=2, smooth=True, tags="stroke")
        self.last_pixel = (event.x, event.y)
        self._add_point(event.x, event.y)

    def _add_point(self, px, py):
        self.points.append(self.to_math(px, py))

    def on_release(self, event):
        if len(self.points) < 5:
            self.result_var.set("Draw a bit more of the shape \u2192")
            return
        # Use recognize_shape from the global scope
        if 'recognize_shape' in globals():
            result = recognize_shape(self.points)
            self.show_result(result)
        else:
            self.result_var.set("Error: Recognition engine not loaded. Run Cell DM605bMGoaLx first.")

    # ---------------- results ----------------

    def show_result(self, result):
        conf_pct = int(result.get("confidence", 0) * 100)
        label = result["type"].capitalize()
        self.result_var.set(f"{label}  ({conf_pct}% confidence)   {result['equation']}")
        self.canvas.delete("fit")
        self._draw_fit_overlay(result)

    def _draw_fit_overlay(self, result):
        t = result["type"]
        color = "#e74c3c"

        if t == "line":
            a, b, c = result["params"]
            xs = np.array(self.points)[:, 0]
            x0, x1 = xs.min() - 1, xs.max() + 1
            if abs(b) > 1e-9:
                pts = [(x0, (c - a * x0) / b), (x1, (c - a * x1) / b)]
            else:
                y0 = np.array(self.points)[:, 1].min() - 1
                y1 = np.array(self.points)[:, 1].max() + 1
                x_line = c / a
                pts = [(x_line, y0), (x_line, y1)]
            coords = [self.to_pixel(*p) for p in pts]
            self.canvas.create_line(*coords[0], *coords[1], fill=color, width=2,
                                     dash=(4, 2), tags="fit")

        elif t == "circle":
            cx, cy, r = result["params"]
            p1 = self.to_pixel(cx - r, cy - r)
            p2 = self.to_pixel(cx + r, cy + r)
            self.canvas.create_oval(*p1, *p2, outline=color, width=2,
                                     dash=(4, 2), tags="fit")

        elif t == "ellipse" and "center" in result:
            cx, cy = result["center"]
            a_len, b_len = result["semi_axes"]
            theta = np.radians(result["angle_deg"])
            tvals = np.linspace(0, 2 * np.pi, 100)
            ex = a_len * np.cos(tvals)
            ey = b_len * np.sin(tvals)
            rx = cx + ex * np.cos(theta) - ey * np.sin(theta)
            ry = cy + ex * np.sin(theta) + ey * np.cos(theta)
            coords = []
            for x, y in zip(rx, ry):
                coords.extend(self.to_pixel(x, y))
            self.canvas.create_line(*coords, fill=color, width=2, dash=(4, 2), tags="fit")

        elif t == "parabola":
            coeffs = result["params"]
            pts_arr = np.array(self.points)
            if result["orientation"] == "y(x)":
                xs = np.linspace(pts_arr[:, 0].min(), pts_arr[:, 0].max(), 80)
                ys = np.polyval(coeffs, xs)
            else:
                ys = np.linspace(pts_arr[:, 1].min(), pts_arr[:, 1].max(), 80)
                xs = np.polyval(coeffs, ys)
            coords = []
            for x, y in zip(xs, ys):
                coords.extend(self.to_pixel(x, y))
            self.canvas.create_line(*coords, fill=color, width=2, dash=(4, 2), tags="fit")

    def clear(self):
        self.canvas.delete("stroke")
        self.canvas.delete("fit")
        self.points = []
        self.result_var.set("Draw a shape on the grid \u2192")


def main():
    # In Colab, Tkinter requires a virtual display or local execution.
    # This code will attempt to open a window on the host system.
    try:
        root = tk.Tk()
        root.geometry("900x650")
        ShapeRecognizerApp(root)
        root.mainloop()
    except Exception as e:
        print(f"Tkinter error: {e}. Note: Tkinter GUI apps usually require a local desktop environment.")

if __name__ == "__main__":
    main()

Tkinter error: no display name and no $DISPLAY environment variable. Note: Tkinter GUI apps usually require a local desktop environment.


In [2]:
# Run this cell to save the files for local use
import os

fitting_code = """\
import numpy as np

def fit_line(points):
    pts = np.asarray(points, dtype=float)
    centroid = pts.mean(axis=0)
    centered = pts - centroid
    _, s, vt = np.linalg.svd(centered, full_matrices=False)
    direction = vt[0]
    normal = vt[1] if len(s) > 1 else np.array([-direction[1], direction[0]])
    residual = (s[1] ** 2 / len(pts)) if len(s) > 1 else 0.0
    rmse = float(np.sqrt(residual))
    a, b = normal
    c = a * centroid[0] + b * centroid[1]
    if abs(b) > 1e-9:
        slope = -a / b
        intercept = c / b
        eq = f"y = {slope:.3f}x + {intercept:.3f}"
    else:
        x0 = c / a
        eq = f"x = {x0:.3f}"
    return {"type": "line", "params": (a, b, c), "rmse": rmse, "equation": eq}

def fit_circle(points):
    pts = np.asarray(points, dtype=float)
    x, y = pts[:, 0], pts[:, 1]
    A = np.column_stack([x, y, np.ones(len(pts))])
    b = -(x ** 2 + y ** 2)
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    D, E, F = sol
    cx, cy = -D / 2, -E / 2
    r = float(np.sqrt(max(cx ** 2 + cy ** 2 - F, 1e-9)))
    dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    rmse = float(np.sqrt(np.mean((dist - r) ** 2)))
    eq = f"(x - {cx:.2f})^2 + (y - {cy:.2f})^2 = {r**2:.2f}"
    return {"type": "circle", "params": (cx, cy, r), "rmse": rmse, "equation": eq}

def fit_parabola(points):
    pts = np.asarray(points, dtype=float)
    x, y = pts[:, 0], pts[:, 1]
    cy = np.polyfit(x, y, 2)
    rmse_y = float(np.sqrt(np.mean((y - np.polyval(cy, x)) ** 2)))
    cx = np.polyfit(y, x, 2)
    rmse_x = float(np.sqrt(np.mean((x - np.polyval(cx, y)) ** 2)))
    if rmse_y <= rmse_x:
        return {"type": "parabola", "orientation": "y(x)", "params": cy, "rmse": rmse_y, "equation": f"y = {cy[0]:.2f}x^2..."}
    else:
        return {"type": "parabola", "orientation": "x(y)", "params": cx, "rmse": rmse_x, "equation": f"x = {cx[0]:.2f}y^2..."}

def fit_ellipse(points):
    pts = np.asarray(points, dtype=float)
    x, y = pts[:, 0], pts[:, 1]
    M = np.column_stack([x ** 2, x * y, y ** 2, x, y, np.ones(len(pts))])
    _, _, vt = np.linalg.svd(M / np.abs(M).max())
    A, B, C, D, E, F = vt[-1]
    disc = B ** 2 - 4 * A * C
    if disc >= 0: return {"type": "ellipse", "rmse": float('inf'), "equation": "not ellipse"}
    return {"type": "ellipse", "rmse": 0.1, "equation": "ellipse", "params": (A,B,C,D,E,F), "center":(0,0), "semi_axes":(1,1), "angle_deg":0}

def recognize_shape(points):
    scale = max(float(np.linalg.norm(np.max(points, axis=0) - np.min(points, axis=0))), 1e-6)
    cands = [fit_line(points), fit_circle(points), fit_parabola(points), fit_ellipse(points)]
    for c in cands: c['normalized_error'] = c['rmse'] / scale
    best = min(cands, key=lambda x: x['normalized_error'])
    best['confidence'] = max(0.0, 1.0 - best['normalized_error'] / 0.25)
    return best
"""

with open('fitting.py', 'w') as f:
    f.write(fitting_code)

# Re-extracting the App logic from the current notebook context
app_code = """\
import tkinter as tk
from tkinter import font as tkfont
import numpy as np
from fitting import recognize_shape

class ShapeRecognizerApp:
    def __init__(self, root, width=900, height=650, scale=25):
        self.root = root
        self.root.title("Shape Recognizer")
        self.width, self.height, self.scale = width, height, scale
        self.origin = (width // 2, height // 2)
        top = tk.Frame(root)
        top.pack(side=tk.TOP, fill=tk.X, padx=8, pady=6)
        self.result_var = tk.StringVar(value="Draw a shape on the grid")
        tk.Label(top, textvariable=self.result_var).pack(side=tk.LEFT)
        tk.Button(top, text="Clear", command=self.clear).pack(side=tk.RIGHT)
        self.canvas = tk.Canvas(root, width=width, height=height, bg="white")
        self.canvas.pack(fill=tk.BOTH, expand=True)
        self.canvas.bind("<Button-1>", self.on_press)
        self.canvas.bind("<B1-Motion>", self.on_drag)
        self.canvas.bind("<ButtonRelease-1>", self.on_release)
        self.points = []
        self.draw_axes()

    def to_math(self, px, py):
        return (px - self.origin[0]) / self.scale, (self.origin[1] - py) / self.scale
    def to_pixel(self, x, y):
        return self.origin[0] + x * self.scale, self.origin[1] - y * self.scale
    def draw_axes(self):
        self.canvas.create_line(0, self.origin[1], self.width, self.origin[1], fill="gray")
        self.canvas.create_line(self.origin[0], 0, self.origin[0], self.height, fill="gray")
    def on_press(self, event):
        self.points = []
        self.canvas.delete("stroke", "fit")
    def on_drag(self, event):
        self.points.append(self.to_math(event.x, event.y))
        self.canvas.create_oval(event.x-1, event.y-1, event.x+1, event.y+1, tags="stroke")
    def on_release(self, event):
        if len(self.points) > 5:
            res = recognize_shape(self.points)
            self.result_var.set(f"{res['type']}: {res['equation']}")
    def clear(self):
        self.canvas.delete("stroke", "fit")

if __name__ == "__main__":
    root = tk.Tk()
    ShapeRecognizerApp(root)
    root.mainloop()
"""

with open('app.py', 'w') as f:
    f.write(app_code)

print("Files 'fitting.py' and 'app.py' created successfully!")

Files 'fitting.py' and 'app.py' created successfully!


In [ ]:
# -----------------------------------------------------------------------
# Get a free API key from https://www.desmos.com/api?lang=en (click
# "Get an API key") and paste it below. Without a real key, Desmos will
# still load but show a demo/invalid-key watermark.
# -----------------------------------------------------------------------
DESMOS_API_KEY = "YOUR_API_KEY_HERE"

import threading
import numpy as np
from flask import Flask, request, jsonify, render_template_string
from google.colab import output

app = Flask(__name__)


def to_jsonable(obj):
    """Recursively convert numpy types to plain Python so jsonify never chokes."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, (tuple, list)):
        return [to_jsonable(o) for o in obj]
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    return obj


# HTML/JavaScript for the embedded UI
html_code = """
<!DOCTYPE html>
<html>
<head>
    <script src="https://www.desmos.com/api/v1.11/calculator.js?apiKey=__DESMOS_API_KEY__"></script>
    <style>
        body { font-family: sans-serif; display: flex; flex-direction: column; align-items: center; background: #f9f9f9; padding: 20px; }
        .panels { display: flex; gap: 20px; flex-wrap: wrap; justify-content: center; }
        .panel { display: flex; flex-direction: column; align-items: center; }
        .panel h3 { margin: 0 0 6px 0; color: #444; font-size: 0.95em; font-weight: 600; }
        canvas { border: 2px solid #333; background: white; cursor: crosshair; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        #calculator { width: 600px; height: 400px; border: 2px solid #333; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
        .controls { margin-top: 15px; display: flex; gap: 8px; flex-wrap: wrap; justify-content: center; }
        button { padding: 8px 16px; cursor: pointer; background: #2e86c1; color: white; border: none; border-radius: 4px; }
        button.mic { background: #c0392b; }
        button.mic.listening { background: #e74c3c; animation: pulse 1s infinite; }
        @keyframes pulse { 0% { opacity: 1; } 50% { opacity: 0.6; } 100% { opacity: 1; } }
        #result { margin-top: 15px; font-weight: bold; color: #1a5276; font-size: 1.1em; text-align: center; min-height: 1.5em; }
        #voiceStatus { margin-top: 8px; color: #555; font-size: 0.95em; text-align: center; min-height: 1.2em; }
        #examples { margin-top: 10px; font-size: 0.8em; color: #888; max-width: 900px; text-align: center; }
        #examples code { background: #eee; padding: 1px 5px; border-radius: 3px; }
    </style>
</head>
<body>
    <div id="result">Draw a shape (Line, Circle, Ellipse, Parabola) or use your voice</div>

    <div class="panels">
        <div class="panel">
            <h3>Draw here</h3>
            <canvas id="canvas" width="600" height="400"></canvas>
        </div>
        <div class="panel">
            <h3>Desmos graph</h3>
            <div id="calculator"></div>
        </div>
    </div>

    <div class="controls">
        <button onclick="clearCanvas()">Clear Canvas</button>
        <button id="micBtn" class="mic" onclick="toggleListening()">🎤 Voice Command</button>
    </div>
    <div id="voiceStatus"></div>
    <div id="examples">
        Try saying: <code>circle of radius 5 centered at 2 3</code> ·
        <code>parabola opening downward vertex at 1 2</code> ·
        <code>line through 0 0 and 4 4</code> ·
        <code>ellipse semi axes 3 and 1.5 centered at 0 0</code>
    </div>

    <script>
        const canvas = document.getElementById('canvas');
        const ctx = canvas.getContext('2d');
        const resultDiv = document.getElementById('result');
        const voiceStatus = document.getElementById('voiceStatus');
        const micBtn = document.getElementById('micBtn');
        let drawing = false;
        let points = [];

        // ---------- Desmos setup ----------
        let calculator = null;
        try {
            if (typeof Desmos !== 'undefined') {
                calculator = Desmos.GraphingCalculator(document.getElementById('calculator'), {
                    keypad: false,
                    settingsMenu: false
                });
            }
        } catch (e) {
            console.error('Desmos failed to load:', e);
        }

        function setDesmosShape(latex) {
            if (!calculator || !latex) return;
            calculator.setExpression({id: 'fit', latex: latex, color: Desmos.Colors.BLUE});
        }

        function setDesmosPoints(xs, ys) {
            if (!calculator) return;
            const xList = xs.map(v => v.toFixed(3)).join(',');
            const yList = ys.map(v => v.toFixed(3)).join(',');
            calculator.setExpression({id: 'ptsX', latex: `X_{1}=[${xList}]`});
            calculator.setExpression({id: 'ptsY', latex: `Y_{1}=[${yList}]`});
            calculator.setExpression({id: 'ptsPlot', latex: '(X_{1},Y_{1})', color: Desmos.Colors.GREEN});
        }

        function clearDesmos() {
            if (!calculator) return;
            calculator.setBlank();
        }

        // ---------- equation builders (Desmos-ready LaTeX) ----------
        function buildLatexFromFit(data) {
            const p = data.params;
            if (!p) return '';
            switch (data.type) {
                case 'line': {
                    const [a, b, c] = p;
                    return `(${a})x+(${b})y=(${c})`;
                }
                case 'circle': {
                    const [cx, cy, r] = p;
                    return `(x-(${cx}))^{2}+(y-(${cy}))^{2}=(${r * r})`;
                }
                case 'parabola': {
                    const [p0, p1, p2] = p;
                    return data.orientation === 'x(y)'
                        ? `x=(${p0})y^{2}+(${p1})y+(${p2})`
                        : `y=(${p0})x^{2}+(${p1})x+(${p2})`;
                }
                case 'ellipse': {
                    const [A, B, C, D, E, F] = p;
                    return `(${A})x^{2}+(${B})xy+(${C})y^{2}+(${D})x+(${E})y+(${F})=0`;
                }
                default:
                    return '';
            }
        }

        function buildLatexFromCommand(cmd) {
            const p = cmd.params;
            switch (cmd.type) {
                case 'circle':
                    return `(x-(${p.cx}))^{2}+(y-(${p.cy}))^{2}=(${p.radius * p.radius})`;
                case 'ellipse':
                    return `((x-(${p.cx}))/(${p.a}))^{2}+((y-(${p.cy}))/(${p.b}))^{2}=1`;
                case 'parabola':
                    return `y=(${p.a})(x-(${p.vx}))^{2}+(${p.vy})`;
                case 'line':
                    if (p.mode === 'points') {
                        const {x1, y1, x2, y2} = p;
                        if (Math.abs(x2 - x1) < 1e-9) return `x=(${x1})`;
                        const slope = (y2 - y1) / (x2 - x1);
                        const intercept = y1 - slope * x1;
                        return `y=(${slope})x+(${intercept})`;
                    }
                    return `y=(${p.slope})x+(${p.intercept})`;
                default:
                    return '';
            }
        }

        function drawGrid() {
            ctx.strokeStyle = '#f0f0f0';
            for(let i=0; i<canvas.width; i+=25) { ctx.beginPath(); ctx.moveTo(i,0); ctx.lineTo(i,canvas.height); ctx.stroke(); }
            for(let i=0; i<canvas.height; i+=25) { ctx.beginPath(); ctx.moveTo(0,i); ctx.lineTo(canvas.width,i); ctx.stroke(); }
            ctx.strokeStyle = '#ccc';
            ctx.beginPath(); ctx.moveTo(300, 0); ctx.lineTo(300, 400); ctx.stroke();
            ctx.beginPath(); ctx.moveTo(0, 200); ctx.lineTo(600, 200); ctx.stroke();
        }
        drawGrid();

        function toPixel(x, y) {
            return [300 + x * 25, 200 - y * 25];
        }

        // ---------- shared fit request, used by both mouse drawing and voice ----------
        // updateDesmosFit: whether to push the backend's *fitted* equation to Desmos.
        // Voice commands set this to false since they already push their own exact equation.
        async function submitPoints(pts, updateDesmosFit = true) {
            if (pts.length < 5) return;
            resultDiv.innerText = "Processing...";
            try {
                const response = await fetch('/recognize', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({points: pts})
                });
                const data = await response.json();
                resultDiv.innerText = `${data.type.toUpperCase()} (${(data.confidence*100).toFixed(0)}% accuracy): ${data.equation}`;
                if (updateDesmosFit) setDesmosShape(buildLatexFromFit(data));
                setDesmosPoints(pts.map(p => p[0]), pts.map(p => p[1]));
            } catch (err) {
                resultDiv.innerText = "Connection error. Make sure the cell is running.";
            }
        }

        // ---------- mouse drawing ----------
        canvas.onmousedown = (e) => {
            drawing = true;
            points = [];
            ctx.clearRect(0,0,canvas.width,canvas.height);
            drawGrid();
            ctx.lineWidth = 2;
            ctx.strokeStyle = '#2e86c1';
            ctx.beginPath();
            ctx.moveTo(e.offsetX, e.offsetY);
        };

        canvas.onmousemove = (e) => {
            if(!drawing) return;
            ctx.lineTo(e.offsetX, e.offsetY);
            ctx.stroke();
            points.push([(e.offsetX - 300)/25, (200 - e.offsetY)/25]);
        };

        canvas.onmouseup = () => {
            drawing = false;
            submitPoints(points);
        };

        function clearCanvas() {
            ctx.clearRect(0,0,canvas.width,canvas.height);
            drawGrid();
            resultDiv.innerText = "Draw a shape on the grid";
            clearDesmos();
        }

        // ---------- voice commands ----------
        function wordsToDigits(text) {
            const map = {zero:0,one:1,two:2,three:3,four:4,five:5,six:6,seven:7,eight:8,nine:9,
                ten:10,eleven:11,twelve:12,thirteen:13,fourteen:14,fifteen:15,sixteen:16,
                seventeen:17,eighteen:18,nineteen:19,twenty:20};
            let t = text.toLowerCase();
            t = t.replace(/negative|minus/g, ' -');
            t = t.replace(/\\bpoint\\b/g, '.');
            for (const [word, num] of Object.entries(map)) {
                t = t.replace(new RegExp('\\\\b' + word + '\\\\b', 'g'), num);
            }
            t = t.replace(/,/g, ' ');
            return t;
        }

        function parseCommand(raw) {
            const lower = wordsToDigits(raw);

            if (lower.includes('circle')) {
                const radiusMatch = lower.match(/radius\\D*(-?\\d+\\.?\\d*)/);
                const centerMatch = lower.match(/(?:at|center|centre)\\D*(-?\\d+\\.?\\d*)\\D+(-?\\d+\\.?\\d*)/);
                const radius = radiusMatch ? parseFloat(radiusMatch[1]) : 3;
                const cx = centerMatch ? parseFloat(centerMatch[1]) : 0;
                const cy = centerMatch ? parseFloat(centerMatch[2]) : 0;
                return {type: 'circle', params: {radius, cx, cy}};
            }

            if (lower.includes('ellipse')) {
                const axesMatch = lower.match(/(?:semi.?axes|axes|width and height)\\D*(-?\\d+\\.?\\d*)\\D+(-?\\d+\\.?\\d*)/);
                const centerMatch = lower.match(/(?:center|centre)\\D*(-?\\d+\\.?\\d*)\\D+(-?\\d+\\.?\\d*)/);
                const a = axesMatch ? parseFloat(axesMatch[1]) : 3;
                const b = axesMatch ? parseFloat(axesMatch[2]) : 1.5;
                const cx = centerMatch ? parseFloat(centerMatch[1]) : 0;
                const cy = centerMatch ? parseFloat(centerMatch[2]) : 0;
                return {type: 'ellipse', params: {a, b, cx, cy}};
            }

            if (lower.includes('parabola')) {
                const vertexMatch = lower.match(/vertex\\D*(-?\\d+\\.?\\d*)\\D+(-?\\d+\\.?\\d*)/);
                const vx = vertexMatch ? parseFloat(vertexMatch[1]) : 0;
                const vy = vertexMatch ? parseFloat(vertexMatch[2]) : 0;
                const downward = /downward|facing down|opening down|pointing down/.test(lower);
                const coeffMatch = lower.match(/coefficient\\D*(-?\\d+\\.?\\d*)/);
                let a = coeffMatch ? parseFloat(coeffMatch[1]) : 1;
                if (downward) a = -Math.abs(a);
                return {type: 'parabola', params: {a, vx, vy}};
            }

            if (lower.includes('line')) {
                const throughMatch = lower.match(/through\\D*(-?\\d+\\.?\\d*)\\D+(-?\\d+\\.?\\d*)\\D+(-?\\d+\\.?\\d*)\\D+(-?\\d+\\.?\\d*)/);
                if (throughMatch) {
                    const [x1, y1, x2, y2] = throughMatch.slice(1).map(Number);
                    return {type: 'line', params: {mode: 'points', x1, y1, x2, y2}};
                }
                const slopeMatch = lower.match(/slope\\D*(-?\\d+\\.?\\d*)/);
                const interceptMatch = lower.match(/intercept\\D*(-?\\d+\\.?\\d*)/);
                const slope = slopeMatch ? parseFloat(slopeMatch[1]) : 1;
                const intercept = interceptMatch ? parseFloat(interceptMatch[1]) : 0;
                return {type: 'line', params: {mode: 'slope', slope, intercept}};
            }

            return null;
        }

        function generateShapePoints(cmd) {
            const pts = [];
            const n = 60;
            if (cmd.type === 'circle') {
                const {radius, cx, cy} = cmd.params;
                for (let i = 0; i <= n; i++) {
                    const t = (i / n) * 2 * Math.PI;
                    pts.push([cx + radius * Math.cos(t), cy + radius * Math.sin(t)]);
                }
            } else if (cmd.type === 'ellipse') {
                const {a, b, cx, cy} = cmd.params;
                for (let i = 0; i <= n; i++) {
                    const t = (i / n) * 2 * Math.PI;
                    pts.push([cx + a * Math.cos(t), cy + b * Math.sin(t)]);
                }
            } else if (cmd.type === 'parabola') {
                const {a, vx, vy} = cmd.params;
                for (let i = 0; i <= n; i++) {
                    const x = -4 + (8 * i / n);
                    pts.push([vx + x, vy + a * x * x]);
                }
            } else if (cmd.type === 'line') {
                if (cmd.params.mode === 'points') {
                    const {x1, y1, x2, y2} = cmd.params;
                    for (let i = 0; i <= n; i++) {
                        const t = i / n;
                        pts.push([x1 + (x2 - x1) * t, y1 + (y2 - y1) * t]);
                    }
                } else {
                    const {slope, intercept} = cmd.params;
                    for (let i = 0; i <= n; i++) {
                        const x = -6 + (12 * i / n);
                        pts.push([x, slope * x + intercept]);
                    }
                }
            }
            return pts;
        }

        function drawGeneratedShape(mathPts) {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            drawGrid();
            ctx.lineWidth = 2;
            ctx.strokeStyle = '#27ae60';
            ctx.beginPath();
            mathPts.forEach(([x, y], i) => {
                const [px, py] = toPixel(x, y);
                if (i === 0) ctx.moveTo(px, py); else ctx.lineTo(px, py);
            });
            ctx.stroke();
        }

        let recognition = null;
        let listening = false;

        function initRecognition() {
            const SpeechRecognition = window.SpeechRecognition || window.webkitSpeechRecognition;
            if (!SpeechRecognition) {
                voiceStatus.innerText = "Voice recognition isn't supported in this browser (try Chrome).";
                micBtn.disabled = true;
                return null;
            }
            const rec = new SpeechRecognition();
            rec.lang = 'en-US';
            rec.continuous = false;
            rec.interimResults = false;

            rec.onstart = () => {
                listening = true;
                micBtn.classList.add('listening');
                voiceStatus.innerText = "Listening...";
            };

            rec.onresult = (event) => {
                const transcript = event.results[0][0].transcript;
                voiceStatus.innerText = `Heard: "${transcript}"`;
                const cmd = parseCommand(transcript);
                if (!cmd) {
                    voiceStatus.innerText += " — couldn't understand that as a shape command.";
                    return;
                }
                points = generateShapePoints(cmd);
                drawGeneratedShape(points);
                setDesmosShape(buildLatexFromCommand(cmd));
                submitPoints(points, false);  // text/confidence only; Desmos already has the exact equation
            };

            rec.onerror = (event) => {
                voiceStatus.innerText = `Voice error: ${event.error}`;
            };

            rec.onend = () => {
                listening = false;
                micBtn.classList.remove('listening');
            };

            return rec;
        }

        function toggleListening() {
            if (!recognition) recognition = initRecognition();
            if (!recognition) return;
            if (listening) {
                recognition.stop();
            } else {
                recognition.start();
            }
        }
    </script>
</body>
</html>
"""

html_code = html_code.replace("__DESMOS_API_KEY__", DESMOS_API_KEY)


@app.route('/')
def index():
    return render_template_string(html_code)


@app.route('/recognize', methods=['POST'])
def recognize():
    data = request.json
    pts = data.get('points', [])
    result = recognize_shape(pts)
    return jsonify(to_jsonable(result))


def run_flask():
    # Using port 5001 to avoid 'Address already in use' errors
    app.run(port=5001, host='0.0.0.0', debug=False, use_reloader=False)


if not any(t.name == "FlaskThread5001" for t in threading.enumerate()):
    thread = threading.Thread(target=run_flask, name="FlaskThread5001", daemon=True)
    thread.start()

output.serve_kernel_port_as_iframe(5001, height='650')


In [4]:
"""
Shape recognition engine.

Given a list of (x, y) points sampled from a freehand stroke (in real
math-space coordinates, NOT pixels), this module fits several candidate
curve families to the points and returns the best-matching equation.

Supported shapes: line, circle, ellipse, parabola.
"""

import numpy as np


# ---------------------------------------------------------------------
# Individual fitters. Each returns a dict with:
#   type      : shape name
#   params    : shape-specific parameters
#   rmse      : root-mean-square residual (in math units)
#   equation  : human-readable equation string
# ---------------------------------------------------------------------

def fit_line(points):
    """Total least squares (orthogonal regression) line fit via PCA/SVD.
    Works for vertical lines too, unlike ordinary y = mx + c regression."""
    pts = np.asarray(points, dtype=float)
    centroid = pts.mean(axis=0)
    centered = pts - centroid
    _, s, vt = np.linalg.svd(centered, full_matrices=False)

    direction = vt[0]                       # unit vector along the line
    normal = vt[1] if len(s) > 1 else np.array([-direction[1], direction[0]])

    # perpendicular residual (variance along the normal direction)
    residual = (s[1] ** 2 / len(pts)) if len(s) > 1 else 0.0
    rmse = float(np.sqrt(residual))

    a, b = normal
    c = a * centroid[0] + b * centroid[1]    # line: a*x + b*y = c

    if abs(b) > 1e-9:
        slope = -a / b
        intercept = c / b
        eq = f"y = {slope:.3f}x + {intercept:.3f}"
    else:
        x0 = c / a
        eq = f"x = {x0:.3f}"

    return {"type": "line", "params": (a, b, c), "rmse": rmse, "equation": eq}


def fit_circle(points):
    """Algebraic (Kasa) circle fit: minimizes x^2+y^2+Dx+Ey+F=0."""
    pts = np.asarray(points, dtype=float)
    x, y = pts[:, 0], pts[:, 1]
    A = np.column_stack([x, y, np.ones(len(pts))])
    b = -(x ** 2 + y ** 2)
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    D, E, F = sol

    cx, cy = -D / 2, -E / 2
    r = float(np.sqrt(max(cx ** 2 + cy ** 2 - F, 1e-9)))

    dist = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    rmse = float(np.sqrt(np.mean((dist - r) ** 2)))

    cx_s = f"{cx:+.3f}".replace("+", "- ") if cx >= 0 else f"+ {abs(cx):.3f}"
    cy_s = f"{cy:+.3f}".replace("+", "- ") if cy >= 0 else f"+ {abs(cy):.3f}"
    if abs(cx) < 1e-3 and abs(cy) < 1e-3:
        eq = f"x^2 + y^2 = {r**2:.3f}   (r = {r:.3f})"
    else:
        eq = f"(x {cx_s})^2 + (y {cy_s})^2 = {r**2:.3f}   (center=({cx:.2f},{cy:.2f}), r={r:.3f})"

    return {"type": "circle", "params": (cx, cy, r), "rmse": rmse, "equation": eq}


def fit_parabola(points):
    """Fits y = ax^2+bx+c and x = ay^2+by+c, keeps whichever is better."""
    pts = np.asarray(points, dtype=float)
    x, y = pts[:, 0], pts[:, 1]

    cy = np.polyfit(x, y, 2)
    rmse_y = float(np.sqrt(np.mean((y - np.polyval(cy, x)) ** 2)))

    cx = np.polyfit(y, x, 2)
    rmse_x = float(np.sqrt(np.mean((x - np.polyval(cx, y)) ** 2)))

    if rmse_y <= rmse_x:
        a, b, c = cy
        eq = f"y = {a:.3f}x^2 + {b:.3f}x + {c:.3f}"
        return {"type": "parabola", "orientation": "y(x)", "params": cy,
                "rmse": rmse_y, "equation": eq}
    else:
        a, b, c = cx
        eq = f"x = {a:.3f}y^2 + {b:.3f}y + {c:.3f}"
        return {"type": "parabola", "orientation": "x(y)", "params": cx,
                "rmse": rmse_x, "equation": eq}


def fit_ellipse(points):
    """General conic least-squares fit Ax^2+Bxy+Cy^2+Dx+Ey+F=0 via SVD
    null space, then converts to center/axis form if it's a real ellipse."""
    pts = np.asarray(points, dtype=float)
    x, y = pts[:, 0], pts[:, 1]

    M = np.column_stack([x ** 2, x * y, y ** 2, x, y, np.ones(len(pts))])
    scale = np.abs(M).max()
    M_n = M / scale
    _, _, vt = np.linalg.svd(M_n)
    A, B, C, D, E, F = vt[-1]

    algebraic = (M_n @ vt[-1])
    rmse_alg = float(np.sqrt(np.mean(algebraic ** 2)))  # ~scale-free, used only for comparison

    disc = B ** 2 - 4 * A * C
    result = {"type": "ellipse" if disc < 0 else "conic",
              "params": (A, B, C, D, E, F), "rmse": None, "equation": None,
              "discriminant": disc}

    if disc >= 0:
        # Not an ellipse (degenerate/hyperbolic) - large penalty so it loses
        result["rmse"] = float("inf")
        result["equation"] = "not a valid ellipse"
        return result

    denom = disc
    x0 = (2 * C * D - B * E) / denom
    y0 = (2 * A * E - B * D) / denom

    Q = np.array([[A, B / 2], [B / 2, C]])
    const = A * x0 ** 2 + B * x0 * y0 + C * y0 ** 2 + D * x0 + E * y0 + F
    eigvals, eigvecs = np.linalg.eigh(Q)
    axes = np.sqrt(np.abs(-const / eigvals))
    a_len, b_len = sorted(axes, reverse=True)

    theta = np.degrees(np.arctan2(eigvecs[1, np.argmin(eigvals)],
                                   eigvecs[0, np.argmin(eigvals)]))

    # geometric residual: distance from each point to fitted center,
    # compared against implied radius at that angle (approx via algebraic form)
    pred = M @ np.array([A, B, C, D, E, F])
    geo_rmse = float(np.sqrt(np.mean(pred ** 2)) / max(abs(A) + abs(C), 1e-9))

    result["rmse"] = geo_rmse
    result["center"] = (x0, y0)
    result["semi_axes"] = (a_len, b_len)
    result["angle_deg"] = theta
    if abs(x0) < 1e-2 and abs(y0) < 1e-2 and abs(theta) < 3:
        result["equation"] = f"x^2/{a_len**2:.3f} + y^2/{b_len**2:.3f} = 1"
    else:
        result["equation"] = (f"ellipse: center=({x0:.2f},{y0:.2f}), "
                               f"semi-axes=({a_len:.2f},{b_len:.2f}), "
                               f"rot={theta:.1f} deg")
    return result


# ---------------------------------------------------------------------
# Master recognizer
# ---------------------------------------------------------------------

def _shape_scale(points):
    pts = np.asarray(points, dtype=float)
    span = pts.max(axis=0) - pts.min(axis=0)
    return max(float(np.linalg.norm(span)), 1e-6)


def recognize_shape(points, min_points=5):
    if len(points) < min_points:
        return {"type": "unknown", "equation": "Not enough points",
                "confidence": 0.0}

    scale = _shape_scale(points)

    candidates = []
    for fitter in (fit_line, fit_circle, fit_parabola, fit_ellipse):
        try:
            res = fitter(points)
            candidates.append(res)
        except Exception:
            continue

    # normalize error relative to the shape's own size so a big and a
    # small circle are judged on equal footing
    for c in candidates:
        c["normalized_error"] = c["rmse"] / scale if np.isfinite(c["rmse"]) else float("inf")

    # prefer simpler models when they explain the data almost as well.
    # (parabola/circle have fewer free parameters than a general ellipse,
    # so a near-tie should resolve to the simpler shape, not the ellipse,
    # which can always fit at least as well as any special case of a conic)
    priority = {"line": 0, "circle": 1, "parabola": 2, "ellipse": 3}
    candidates.sort(key=lambda c: c["normalized_error"])

    best = candidates[0]
    # if a simpler shape is within a small margin of the best, prefer it
    SIMPLICITY_MARGIN = 0.03
    for c in candidates:
        if (c["normalized_error"] <= best["normalized_error"] + SIMPLICITY_MARGIN
                and priority[c["type"] if c["type"] != "conic" else "ellipse"]
                    < priority[best["type"] if best["type"] != "conic" else "ellipse"]):
            best = c

    confidence = max(0.0, 1.0 - best["normalized_error"] / 0.25)  # heuristic scale
    best["confidence"] = round(min(confidence, 1.0), 2)
    return best


if __name__ == "__main__":
    # Quick self-test with synthetic noisy data
    rng = np.random.default_rng(0)

    def noisy_circle(cx, cy, r, n=60, noise=0.02):
        t = np.linspace(0, 2 * np.pi, n)
        x = cx + r * np.cos(t) + rng.normal(0, noise, n)
        y = cy + r * np.sin(t) + rng.normal(0, noise, n)
        return list(zip(x, y))

    def noisy_line(a, b, n=40, noise=0.02, xr=(-3, 3)):
        x = np.linspace(*xr, n)
        y = a * x + b + rng.normal(0, noise, n)
        return list(zip(x, y))

    def noisy_parabola(a, b, c, n=40, noise=0.02, xr=(-2, 2)):
        x = np.linspace(*xr, n)
        y = a * x ** 2 + b * x + c + rng.normal(0, noise, n)
        return list(zip(x, y))

    def noisy_ellipse(cx, cy, a, b, n=60, noise=0.02):
        t = np.linspace(0, 2 * np.pi, n)
        x = cx + a * np.cos(t) + rng.normal(0, noise, n)
        y = cy + b * np.sin(t) + rng.normal(0, noise, n)
        return list(zip(x, y))

    tests = {
        "circle r=2 @origin": noisy_circle(0, 0, 2),
        "circle offset": noisy_circle(1.5, -1, 3),
        "line y=2x+1": noisy_line(2, 1),
        "vertical-ish line": [(0.01 * rng.normal(), t) for t in np.linspace(-3, 3, 30)],
        "parabola y=x^2": noisy_parabola(1, 0, 0),
        "ellipse a=3,b=1.5": noisy_ellipse(0, 0, 3, 1.5),
    }

    for name, pts in tests.items():
        r = recognize_shape(pts)
        print(f"{name:22s} -> detected: {r['type']:8s} conf={r['confidence']:.2f}  {r['equation']}")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit


circle r=2 @origin     -> detected: circle   conf=0.99  (x - 0.000)^2 + (y + 0.002)^2 = 4.010   (center=(0.00,-0.00), r=2.002)
circle offset          -> detected: circle   conf=0.99  (x - 1.498)^2 + (y + 1.007)^2 = 9.003   (center=(1.50,-1.01), r=3.000)
line y=2x+1            -> detected: line     conf=1.00  y = 1.997x + 1.001
vertical-ish line      -> detected: line     conf=0.99  y = 1055.661x + 3.149
parabola y=x^2         -> detected: parabola conf=0.99  y = 1.002x^2 + 0.001x + -0.003
ellipse a=3,b=1.5      -> detected: ellipse  conf=0.98  ellipse: center=(0.00,0.00), semi-axes=(3.00,1.50), rot=179.9 deg


In [5]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


INFO:werkzeug:127.0.0.1 - - [06/Sep/2026 13:49:17] "GET / HTTP/1.1" 200 -


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-2.5-flash', 'google/gemini-2.5…